<a href="https://colab.research.google.com/github/alj-x64/Realtime-Bass-Tablature-Transcription/blob/main/Adaptive_Multi_Stage_HPO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import random
import math
import copy
import csv
import os

class AdaptiveMultiStageHPO:
    def __init__(self, total_budget=30, alpha=0.5, beta=0.5, rho_s=0.60, rho_r=0.40, gamma=0.30, log_file="hpo_history.csv"):
        """
        Inisyalisasyon ng optimizer base sa mathematical constraints ng thesis mo.
        """
        self.B = total_budget
        self.alpha = alpha
        self.beta = beta
        self.log_file = log_file
        
        # Thesis Allocation Ratios (Table 3)
        self.rho_S = rho_s
        self.rho_R = rho_r
        self.gamma = gamma
        
        # Kalkulahin ang Budgets (Equations 9 - 18)
        self.S = int(self.rho_S * self.B)
        self.R = int(self.rho_R * self.B)
        
        # Eq 16: P = ceil(gamma * rho_S * B)
        self.P = max(2, math.ceil(self.gamma * self.S)) 
        
        # Eq 17: G = floor(rho_S * B / P)
        self.G = max(1, math.floor(self.S / self.P))
        
        print(f"--- HPO Budgets Initialized ---")
        print(f"Total Budget (B): {self.B} trials")
        print(f"Selection Budget (S): {self.S} | Refinement Budget (R): {self.R}")
        print(f"Population Size (P): {self.P} | Generations (G): {self.G}")
        print(f"-------------------------------")

        # Search Space (Table 2)
        self.search_space = {
            'learning_rate': {'type': 'continuous', 'range': [1e-5, 1e-2]},
            'dropout_rate': {'type': 'continuous', 'range': [0.1, 0.5]},
            'activation': {'type': 'categorical', 'values': ['ReLU', 'Tanh', 'ELU']},
            'conv_layers': {'type': 'discrete', 'range': [1, 4]},
            'filter_layers': {'type': 'categorical', 'values': [16, 32, 64, 128]},
            'kernel_size': {'type': 'categorical', 'values': [2, 3, 5, 7]}
        }

        # Initialize CSV Logger
        self._init_csv_logger()

    def _init_csv_logger(self):
        """Gumagawa ng CSV header para sa persistent data tracking."""
        file_exists = os.path.isfile(self.log_file)
        with open(self.log_file, mode='w', newline='') as f:
            writer = csv.writer(f)
            # CSV Headers
            writer.writerow([
                'Stage', 'Gen_or_Rank', 'LearningRate', 'DropoutRate', 'Activation', 
                'ConvLayers', 'FilterLayers', 'KernelSize', 'Loss', 'Latency_ms', 
                'Fitness', 'Status'
            ])

    def log_individual(self, stage, gen_or_rank, config, loss, latency, fitness, status):
        """Nagsusulat ng isang row sa CSV file in real-time."""
        with open(self.log_file, mode='a', newline='') as f:
            writer = csv.writer(f)
            writer.writerow([
                stage, gen_or_rank, 
                f"{config['learning_rate']:.6f}", config['dropout_rate'], config['activation'],
                config['conv_layers'], config['filter_layers'], config['kernel_size'],
                f"{loss:.4f}", f"{latency:.2f}", f"{fitness:.4f}", status
            ])

    def generate_individual(self):
        """Gumagawa ng random na hyperparameter configuration vector (Eq 7)."""
        ind = {}
        for key, params in self.search_space.items():
            if params['type'] == 'continuous':
                if key == 'learning_rate':
                    ind[key] = 10 ** np.random.uniform(np.log10(params['range'][0]), np.log10(params['range'][1]))
                else:
                    ind[key] = np.random.uniform(params['range'][0], params['range'][1])
            elif params['type'] == 'discrete':
                ind[key] = np.random.randint(params['range'][0], params['range'][1] + 1)
            elif params['type'] == 'categorical':
                ind[key] = random.choice(params['values'])
        return ind

    def fitness_function(self, loss, latency):
        """Kino-compute ang fitness kung saan balance ang accuracy at latency (Eq 19 at Eq 39)."""
        latency_penalty = (latency / 200.0) if latency > 0 else 0 
        return self.alpha * (1.0 - loss) - self.beta * latency_penalty

    def crossover(self, parent1, parent2):
        """Uniform crossover na may 0.5 probability (Eq 30)."""
        offspring = {}
        for key in self.search_space.keys():
            offspring[key] = parent1[key] if random.random() < 0.5 else parent2[key]
        return offspring

    def mutate(self, individual):
        """Nag-a-apply ng task-dependent mutation na may Pm = 0.5 (Eq 31 - 33)."""
        mutated = copy.deepcopy(individual)
        for key, params in self.search_space.items():
            if random.random() < 0.5: 
                if params['type'] == 'continuous':
                    sigma = (params['range'][1] - params['range'][0]) * 0.1 
                    new_val = mutated[key] + np.random.normal(0, sigma)
                    mutated[key] = np.clip(new_val, params['range'][0], params['range'][1])
                elif params['type'] == 'discrete':
                    mutated[key] = np.random.randint(params['range'][0], params['range'][1] + 1)
                elif params['type'] == 'categorical':
                    mutated[key] = random.choice(params['values'])
        return mutated

    def run_optimization(self, train_eval_fn):
        """Pinapatakbo ang buong Selection at Refinement stages."""
        
        # ==========================================
        # 1. SELECTION STAGE
        # ==========================================
        population = [{'config': self.generate_individual(), 'fitness': 0, 'loss': 0, 'latency': 0} for _ in range(self.P)]
        all_losses = [] 

        for gen in range(self.G):
            print(f"\n--- Generation {gen + 1}/{self.G} ---")
            
            for ind in population:
                loss, latency = train_eval_fn(ind['config'], stress_test=False) 
                ind['loss'] = loss
                ind['latency'] = latency
                ind['fitness'] = self.fitness_function(loss, latency)
                all_losses.append(loss)
                
                # I-log agad-agad sa CSV
                self.log_individual("Selection", f"Gen_{gen+1}", ind['config'], loss, latency, ind['fitness'], "Evaluated")
            
            population.sort(key=lambda x: x['fitness'], reverse=True)
            survivors = population[:max(1, len(population)//2)]
            
            next_generation = copy.deepcopy(survivors)
            while len(next_generation) < self.P:
                p1, p2 = random.sample(survivors, 2) if len(survivors) > 1 else (survivors[0], survivors[0])
                offspring_config = self.crossover(p1['config'], p2['config'])
                mutated_config = self.mutate(offspring_config)
                next_generation.append({'config': mutated_config, 'fitness': 0, 'loss': 0, 'latency': 0})
            
            population = next_generation

        # Evaluate offspring na wala pang score
        for ind in population:
            if ind['fitness'] == 0: 
                loss, latency = train_eval_fn(ind['config'], stress_test=False)
                ind['loss'], ind['latency'] = loss, latency
                ind['fitness'] = self.fitness_function(loss, latency)
                all_losses.append(loss)
                self.log_individual("Selection", "Final_Eval", ind['config'], loss, latency, ind['fitness'], "Evaluated")

        population.sort(key=lambda x: x['fitness'], reverse=True)
        p25_loss = np.percentile(all_losses, 25) 

        # ==========================================
        # 2. REFINEMENT STAGE (Constraint-Aware)
        # ==========================================
        print("\n===========================================")
        print("--- Entering Refinement Stage ---")
        print(f"Strict Criteria -> Loss <= {p25_loss:.4f} (Top 25%), Latency <= 200.0ms")
        print("===========================================")

        for rank in range(min(self.R, len(population))):
            candidate = population[rank]
            print(f"\nTesting Candidate Rank {rank + 1} (Fitness: {candidate['fitness']:.4f})...")
            
            stress_loss, stress_latency = train_eval_fn(candidate['config'], stress_test=True)
            
            p_trigger = 0.10
            if np.random.rand() < p_trigger: 
                jitter = np.random.uniform(0, 50)
                stress_latency += jitter
                print(f"   [!] Binomial Trial Triggered: Added {jitter:.2f}ms Latency Jitter.")
            
            print(f"   Post-Stress Results -> Loss: {stress_loss:.4f}, Latency: {stress_latency:.2f}ms")
            
            # 3. CONSTRAINT-AWARE REFINEMENT (Eq 40)
            if stress_latency <= 200.0 and stress_loss <= p25_loss:
                print(f"\n>>> YES: Accept the individual as the optimal hyperparameter! (Rank {rank + 1}) <<<")
                self.log_individual("Refinement", f"Rank_{rank+1}", candidate['config'], stress_loss, stress_latency, candidate['fitness'], "PROMOTED_OPTIMAL")
                return candidate['config']
            else:
                print(f"   [x] NO: Failed strict constraints.")
                print(f"   [-] Action: kill the individual.")
                print(f"   [>] Action: Get the next best individual (of selection stage) based on the fitness function; promote to refinement stage.")
                self.log_individual("Refinement", f"Rank_{rank+1}", candidate['config'], stress_loss, stress_latency, candidate['fitness'], "ELIMINATED")

        print("\n>>> YES (Budget Used Up): NO CANDIDATE PASSED ALL STRICT CONSTRAINTS. END. <<<")
        self.log_individual("Refinement", "Fallback", population[0]['config'], population[0]['loss'], population[0]['latency'], population[0]['fitness'], "FALLBACK_BEST_EFFORT")
        return population[0]['config']